# Unit 4.1 Exercise - Naïve Bayes

In [1]:
import re
import math
from collections import Counter, defaultdict

documents = [
    ("Free money now!!!", "SPAM"),
    ("Hi mom, how are you?", "HAM"),
    ("Lowest price for your meds", "SPAM"),
    ("Are we still on for dinner?", "HAM"),
    ("Win a free iPhone today", "SPAM"),
    ("Let's catch up tomorrow at the office", "HAM"),
    ("Meeting at 3 PM tomorrow", "HAM"),
    ("Get 50% off, limited time!", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes!", "SPAM"),
    ("Can you send the report?", "HAM")
]

def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())
print("Dataset loaded.")

Dataset loaded.


## A. Generate the Bag of Words

In [2]:
vocab = set()
class_word_counts = defaultdict(Counter)
class_doc_counts = Counter()

for text, label in documents:
    class_doc_counts[label] += 1
    words = tokenize(text)
    vocab.update(words)
    class_word_counts[label].update(words)

print("Vocabulary:")
print(sorted(vocab))

print("\nHAM Bag of Words")
print(class_word_counts["HAM"])

print("\nSPAM Bag of Words")
print(class_word_counts["SPAM"])

Vocabulary:
['3', '50', 'a', 'are', 'at', 'can', 'catch', 'click', 'dinner', 'for', 'free', 'get', 'here', 'hi', 'how', 'in', 'iphone', 'let', 'limited', 'lowest', 'meds', 'meeting', 'mom', 'money', 'now', 'off', 'office', 'on', 'pm', 'price', 'prizes', 'report', 's', 'send', 'still', 'team', 'the', 'time', 'today', 'tomorrow', 'up', 'we', 'win', 'you', 'your']

HAM Bag of Words
Counter({'the': 3, 'are': 2, 'you': 2, 'tomorrow': 2, 'at': 2, 'office': 2, 'meeting': 2, 'hi': 1, 'mom': 1, 'how': 1, 'we': 1, 'still': 1, 'on': 1, 'for': 1, 'dinner': 1, 'let': 1, 's': 1, 'catch': 1, 'up': 1, '3': 1, 'pm': 1, 'team': 1, 'in': 1, 'can': 1, 'send': 1, 'report': 1})

SPAM Bag of Words
Counter({'free': 2, 'for': 2, 'money': 1, 'now': 1, 'lowest': 1, 'price': 1, 'your': 1, 'meds': 1, 'win': 1, 'a': 1, 'iphone': 1, 'today': 1, 'get': 1, '50': 1, 'off': 1, 'limited': 1, 'time': 1, 'click': 1, 'here': 1, 'prizes': 1})


## B. Compute Prior Probabilities

In [3]:
total_docs = len(documents)
priors = {label: count/total_docs for label, count in class_doc_counts.items()}

for label, value in priors.items():
    print(f"{label}: {value:.4f}")

SPAM: 0.4545
HAM: 0.5455


## C. Compute Likelihoods (Laplace Smoothing)

In [4]:
likelihood = defaultdict(dict)
V = len(vocab)

for label in class_word_counts:
    total_words = sum(class_word_counts[label].values())
    for word in vocab:
        likelihood[label][word] = (class_word_counts[label][word] + 1) / (total_words + V)

print("Sample Likelihood Values:")
for word in sorted(list(vocab))[:10]:
    print(word, "HAM =", round(likelihood["HAM"][word],4),
          "SPAM =", round(likelihood["SPAM"][word],4))

Sample Likelihood Values:
3 HAM = 0.0253 SPAM = 0.0149
50 HAM = 0.0127 SPAM = 0.0299
a HAM = 0.0127 SPAM = 0.0299
are HAM = 0.038 SPAM = 0.0149
at HAM = 0.038 SPAM = 0.0149
can HAM = 0.0253 SPAM = 0.0149
catch HAM = 0.0253 SPAM = 0.0149
click HAM = 0.0127 SPAM = 0.0299
dinner HAM = 0.0253 SPAM = 0.0149
for HAM = 0.0253 SPAM = 0.0448


## D. Manual Prediction

In [5]:
def predict(sentence):
    words = tokenize(sentence)
    scores = {}

    for label in priors:
        score = math.log(priors[label])
        total_words = sum(class_word_counts[label].values())

        for word in words:
            if word in vocab:
                score += math.log(likelihood[label][word])
            else:
                score += math.log(1/(total_words+V))

        scores[label] = score

    return max(scores, key=scores.get), scores

tests = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
]

for sentence in tests:
    prediction, scores = predict(sentence)
    print("="*50)
    print(sentence)
    print("Prediction:", prediction)
    print(scores)

Limited offer, click here!
Prediction: SPAM
{'SPAM': -15.527786296248298, 'HAM': -18.083927213438404}
Meeting at 2 PM with the manager.
Prediction: HAM
{'SPAM': -30.221305696101027, 'HAM': -26.91560465182341}


## E. Scikit-Learn Multinomial Naïve Bayes

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

texts = [x[0] for x in documents]
labels = [x[1] for x in documents]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)

model = MultinomialNB()
model.fit(X, labels)

X_test = vectorizer.transform([
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
])

predictions = model.predict(X_test)

print("Scikit-Learn Predictions")
for sentence, pred in zip([
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
], predictions):
    print(sentence)
    print("Prediction:", pred)
    print()

Scikit-Learn Predictions
Limited offer, click here!
Prediction: SPAM

Meeting at 2 PM with the manager.
Prediction: HAM

